# L3d: Comparing Sorting Algorithms and Their Scaling
In this lab, we finish [our own bubble-sort implementation](src/Compute.jl), listen to it work, and then compare its average runtime against a recursive [Quicksort](https://en.wikipedia.org/wiki/Quicksort) implementation and [Julia's built-in sort function](https://docs.julialang.org/en/v1/base/sort/#Base.sort) using [the BenchmarkTools.jl package](https://github.com/JuliaCI/BenchmarkTools.jl).

> __Learning Objectives:__
>
> By the end of this lab, you should be able to:
> * __Finish an implementation against its contract:__ Complete the comparison, swap, and early-exit logic of the mutating bubble sort so it reproduces the reference ordering. An algorithm you completed yourself is one you can reason about when the timing results need explaining.
> * __Test an implementation before timing it:__ Establish that a sorting routine reproduces the reference ordering before measuring how fast it runs, because a quick wrong answer is worth nothing. The tests in this lab are small and cheap, and they are what makes every timing number the lab records worth reporting.
> * __Measure scaling and decide between building and buying:__ Benchmark one operation across growing inputs and read the growth rate separately from the constant factor sitting in front of it. Comparing our own implementations against the library routine turns those curves into a concrete build-versus-buy decision.

<!-- We'll complete four main tasks in this notebook:
- **Task 1**: Complete the swap logic in [our `bubblesort!(...)` function](src/Compute.jl), verify it on a small array, and (optionally) hear it sort.
- **Task 2**: Verify [our `bubblesort(...)` implementation](src/Compute.jl) against [Julia's `sort(...)` function](https://docs.julialang.org/en/v1/base/sort/#Base.sort), then benchmark it across array sizes to establish a baseline for the other algorithms.
- **Task 3**: Verify [our `quicksort(...)` implementation](src/Compute.jl) the same way and benchmark it against that baseline.
- **Task 4**: Benchmark Julia's built-in sort to see how both custom implementations compare against an optimized library routine. -->

Let's get started!
___

## Algorithms
Bubblesort and Quicksort are two different sorting algorithms that have different approaches to sorting a list of elements.

> * __How does Bubblesort work?__ It involves repeatedly passing through the list, comparing adjacent items, and swapping any two neighboring items that are out of order. The algorithm gets its name from the way values rise through the list like bubbles: each pass carries the largest remaining element to its final position at the end of the list, while the smaller elements work toward the front one position at a time. Let's take a look at the Bubblesort algorithm in more detail [here](CHEME-5800-L3d-Algorithm-Bubblesort-Fall-2026.ipynb).
> * __How does Quicksort work?__ Quicksort is a recursive sorting algorithm that works by selecting a pivot element and partitioning the remaining elements according to their value relative to the pivot. The algorithm then recursively sorts the partitions until they have fewer than two elements. The choice of pivot is critical for the algorithm's efficiency. Let's take a look at the Quicksort algorithm [here](CHEME-5800-L3d-Algorithm-Quicksort-Fall-2026.ipynb)


Both are worth timing before we decide which one to reach for.
___

## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file and loading any needed resources.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/).

Let's set up our code environment:

In [ ]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

The course environment also loads [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl); see [the documentation](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/). This lab does not need it. The sorting implementations we benchmark live in [`src/Compute.jl`](src/Compute.jl), and the timing comes from [the BenchmarkTools.jl package](https://github.com/JuliaCI/BenchmarkTools.jl). The bubble-sort tones load through [the WAV.jl package](https://github.com/dancasimiro/WAV.jl), and the `sounds/` folder holds the 128 tone files described in [`sounds/README.md`](sounds/README.md).

### Constants
Before we get started, let's set up some constants. See the comment next to each value for what it is, its permissible values, units, etc.

In [ ]:
max_number_of_trials = 10; # exponent; largest vector has 2^10 elements
number_of_items_per_trial = [2^i for i ∈ 1:max_number_of_trials]; # an array comprehension, yet another iteration pattern
play_sounds = false; # flip to true in class to hear each bubble-sort pass; leave false for silent runs

___

## Task 1: Complete the bubble-sort implementation
The timing comparisons in this lab only mean something if the code being timed is code we understand, so we start by finishing the bubble-sort implementation ourselves. The [`src/Compute.jl`](src/Compute.jl) file ships the `L3dSorting` module with [our `bubblesort!(...)` function](src/Compute.jl) incomplete: it raises an implementation error until you finish it, and the non-mutating [`bubblesort(...)` wrapper](src/Compute.jl) simply copies its input and hands the copy to the mutating version, so completing one function completes both.

> __What to write:__
>
> * __TODO 1:__ Loop over passes `1` through `length(values)`, and at the top of each pass call [the private `_play_sound(...)` helper](src/Sounds.jl) when a sound library was supplied, so the pass is audible.
> * __TODO 2:__ Inside each pass, sweep positions `1` through `length(values) - pass`, where [the `length(...)` function](https://docs.julialang.org/en/v1/base/collections/#Base.length) counts the elements, and swap any neighboring pair that [the `isless(...)` function](https://docs.julialang.org/en/v1/base/base/#Base.isless) says is out of order, the same ordering [Julia's `sort(...)` function](https://docs.julialang.org/en/v1/base/sort/#Base.sort) uses.
> * __TODO 3:__ Track whether the pass swapped anything, stop early when a full pass changes nothing, and return the sorted vector.

The test cell in this task tells you when you are done: it passes when your implementation reproduces the ordering of [Julia's `sort(...)` function](https://docs.julialang.org/en/v1/base/sort/#Base.sort) on the array it draws. Work in the source file and save it, then re-run the setup cell in the Setup, Data, and Prerequisites section so the notebook reloads your edits. The reload is silent, so do not wait for a confirmation message. The notebook calls the module's functions by their qualified names, such as `L3dSorting.bubblesort(...)`, which is what lets the reloaded module take effect without restarting the kernel. Then re-run the test cell.

There is also a payoff for finishing. The `sounds/` folder holds 128 short tones, one per integer value, rising in pitch with the value, and [`sounds/README.md`](sounds/README.md) records their provenance. With the `play_sounds` constant flipped to `true`, the `sound_library::Union{Nothing, Dict{Int64, Tuple{Matrix{Float64}, Float32}}}` variable loads the whole set, and sorting an integer array drawn from `1:128` then plays the array at the top of every pass: a shuffled array sounds jumbled, and a finished sort plays a rising scale. Let's load the library, or stay silent:

In [ ]:
sound_library = play_sounds ? L3dSorting.load_sound_library(joinpath(CHEME5800_L3D_ROOT, "sounds")) : nothing; # 128 tones, or silent

With the implementation complete, the swap logic has to prove itself: a sorted copy must match the reference ordering exactly on the shuffled input we hand it. When the sound library is loaded, this check also plays the array at the top of every pass. Does it pass?

In [ ]:
let
    audible_demo = rand(1:128, 25); # integer values 1:128, so each maps to a tone
    @test L3dSorting.bubblesort(audible_demo; sounds = sound_library) == sort(audible_demo)
end

A green test says the completed implementation agrees with the library on this input. The timing comparisons in this lab lean on that agreement: what we measure is the cost of a correct sort, not a fast wrong one.
___

## Task 2: Establish a performance baseline: Bubblesort
One passing test on one small input is not much evidence, so before timing anything we test the completed implementation again on a larger random input. This time we also check the contracts: [the mutating `bubblesort!(...)` version](src/Compute.jl) must return the very vector it was handed, sorted in place, while the non-mutating wrapper must leave its input untouched. We then establish a performance baseline for comparison with the Quicksort algorithm.

> We'll use [the `Test.jl` package](https://docs.julialang.org/en/v1/stdlib/Test/) to write __unit tests__ for our bubble sort implementation. The [`Test.jl` package](https://docs.julialang.org/en/v1/stdlib/Test/) provides a framework for writing and running tests in Julia, including support for assertions, test cases, and test suites. Let's use [the @test macro](https://docs.julialang.org/en/v1/stdlib/Test/#Test.@test) to check that our bubble sort implementation works correctly.

So, did we pass the tests?

In [ ]:
let

    # initialize -
    N = 1000; # number of elements in the random vector
    arr = rand(N); # random vector length N
    original = copy(arr); # keep a copy, so we can check the wrapper left arr alone

    # check: do we get the same result as the built-in sort function?
    @test sort(arr) == L3dSorting.bubblesort(arr) # if the test fails, an error is thrown!

    # check the contracts: the wrapper must not touch its input ...
    @test arr == original

    # ... and the mutating version must sort the very vector it was handed -
    mutable_demo = rand(N);
    expected = sort(mutable_demo); # sort(...) copies, so this is safe to compute first
    @test L3dSorting.bubblesort!(mutable_demo) === mutable_demo
    @test mutable_demo == expected
end

Okay, so if we get here, all seems to be good with [our `bubblesort(...)` implementation](src/Compute.jl). Let's measure how it performs as we increase the size of the input array, using the [BenchmarkTools.jl package](https://github.com/JuliaCI/BenchmarkTools.jl) to get accurate timing measurements.

The benchmark uses a [`let ... end` block](https://docs.julialang.org/en/v1/manual/variables-and-scoping/#Let-Blocks) to create a local scope and performs the following steps:
1. **Initialize a DataFrame** to store our results with columns for array size (`n`), mean runtime (`μ`), and standard deviation (`σ`) using the [DataFrames.jl package](https://github.com/JuliaData/DataFrames.jl)
2. **Loop through different array sizes** from our `number_of_items_per_trial` array (powers of 2 from $2^{1}$ to $2^{\texttt{max\_number\_of\_trials}}$, which the Constants subsection fixes at $2^{10}$)
3. **For each array size:**
   - Create a benchmark using [the `@benchmarkable` macro](https://juliaci.github.io/BenchmarkTools.jl/stable/reference/#BenchmarkTools.@benchmarkable-Tuple) that will test our `bubblesort` function
   - Use `setup=` to draw a fresh random `Float64` vector for each sample, the same kind of data the test cells sort. The setup code runs before the clock starts, so we never measure data generation
   - Run the benchmark multiple times and collect timing statistics
   - Store the results (array size, mean time, standard deviation) as a row in our DataFrame

The result will be stored in the `bubble_sort_data::DataFrame` variable containing performance data that we can analyze and visualize.

In [ ]:
bubble_sort_data = let
    bubble_sort_data = DataFrame();
    for i ∈ eachindex(number_of_items_per_trial)
        size_of_rand_vec_to_sort = number_of_items_per_trial[i];
    
        # run the test with different size vectors -
        test_run = @benchmarkable L3dSorting.bubblesort(data) setup=(data=rand($(size_of_rand_vec_to_sort)));
        results = run(test_run; samples = 20, seconds = 0.25, evals = 1)
    
        # store the results -
        row = (
            n = size_of_rand_vec_to_sort,
            μ = mean(results.times),
            σ = std(results.times)
        );
        push!(bubble_sort_data, row)
    end
    bubble_sort_data
end

___

## Task 3: Quicksort
Task 2 left us with a quadratic baseline and a curve to beat. Quicksort is the recursive alternative from the lecture, so in this task we verify that our Quicksort implementation works as expected. We also establish the performance of this method in comparison with the Bubblesort algorithm.

> As in Task 2, [the @test macro](https://docs.julialang.org/en/v1/stdlib/Test/#Test.@test) is the check and [Julia's `sort(...)` function](https://docs.julialang.org/en/v1/base/sort/#Base.sort) supplies the reference ordering, so the two implementations are held to the same standard. The input is a random `Float64` vector, so its values are distinct, and [our `quicksort(...)` implementation](src/Compute.jl) is exercised on the ordinary path through the partition rather than on the duplicate-heavy case the algorithm notebook discusses.

Does it work?

In [ ]:
let

    # initialize -
    N = 1000; # number of elements in the random vector
    arr = rand(N); # random vector length N

    # check: do we get the same result as the built-in sort function?
    @test sort(arr) == L3dSorting.quicksort(arr) # if the test fails, an error is thrown!
end

Okay, so if we get here, all seems to be good with our `quicksort(...)` implementation, so let's see how our code performs as we increase the vector size to be sorted. 

Let's use the same benchmarking pattern as we did for the Bubblesort algorithm. The benchmarking results for the Quicksort algorithm will be stored in the `quick_sort_data::DataFrame` variable containing performance data that we can analyze and visualize.

In [ ]:
quick_sort_data = let
    
    quick_sort_data = DataFrame();
    for i ∈ eachindex(number_of_items_per_trial)
        size_of_rand_vec_to_sort = number_of_items_per_trial[i];
    
        # run the test with different size vectors -
        test_run = @benchmarkable L3dSorting.quicksort(data) setup=(data=rand($(size_of_rand_vec_to_sort)));
        results = run(test_run; samples = 20, seconds = 0.25, evals = 1)
    
        # store the results -
        row = (
            n = size_of_rand_vec_to_sort,
            μ = mean(results.times),
            σ = std(results.times)
        );
        push!(quick_sort_data, row)
    end
    quick_sort_data
end

___

## Task 4: What is the scaling of the built-in sort function?
Julia provides [several sorting algorithms and picks a default suited to the data](https://docs.julialang.org/en/v1/base/sort/#Sorting-Functions). How do our implementations perform against what Julia can offer? 

> __Should you write your own?__ This is yet another example of the __buy versus build__ conundrum. Should we build our own implementation, or __buy__ someone else's? You should (almost) always __buy__, and benefit from the hard (optimized) work of others. But let's see if that is true in this case.

We'll use the same benchmarking approach as we did for the Bubblesort and Quicksort algorithms. The results will be stored in the `julia_sort_data::DataFrame` variable containing performance data that we can analyze and visualize.

In [ ]:
julia_sort_data = let
    julia_sort_data = DataFrame();
    for i ∈ eachindex(number_of_items_per_trial)
        size_of_rand_vec_to_sort = number_of_items_per_trial[i];
    
        # run the test with different size vectors -
        test_run = @benchmarkable sort(data) setup=(data=rand($(size_of_rand_vec_to_sort)));
        results = run(test_run; samples = 20, seconds = 0.25, evals = 1)
    
        # store the results -
        row = (
            n = size_of_rand_vec_to_sort,
            μ = mean(results.times),
            σ = std(results.times)
        );
        push!(julia_sort_data, row)
    end
    julia_sort_data
end

___

## Visualize
Unhide the code to see how we plotted the average runtime of each sorting method as a function of the length of the vector $n$.

> __What the curves show:__ For very short sequences, a handful of elements, our `bubblesort(...)` runs about as fast as the built-in sort and leaves our `quicksort(...)` far behind: a quadratic algorithm with almost no overhead is competitive when $n$ is small. Once the sequences grow, [Julia's built-in `sort(...)` function](https://docs.julialang.org/en/v1/base/sort/#Sorting-Functions) wins by a wide margin: at $n = 1024$ it beats our `quicksort(...)` implementation by more than an order of magnitude.

Two separate lessons hide in these curves. First, scaling: our `quicksort(...)` grows like its $\mathcal{O}(n\log{n})$ average case predicts, while our `bubblesort(...)` bends upward toward its $\mathcal{O}(n^{2})$ bound, so the gap between them widens without limit. Look at the middle of the plot for the contrasting case: from roughly $n = 16$ to $n = 256$ our `quicksort(...)` and the built-in sort run close to parallel, a fixed multiple apart, which is what a difference in constant factor alone looks like. Second, the built-in curve is not just our quicksort with a smaller constant factor in front: it grows more slowly than $n\log{n}$ over this range because [Julia's `sort(...)`](https://docs.julialang.org/en/v1/base/sort/#Sorting-Functions) does not commit to one algorithm at all. It inspects the element type and the input and dispatches to specialized methods, including [radix-style sorts](https://en.wikipedia.org/wiki/Radix_sort) that order common numeric types without comparing pairs of elements. Constant factors still matter, scaling analysis ignores them but real races do not, yet the deeper reason the library wins here is that it chose a better algorithm for the data than the one we wrote.

In [ ]:
let
    plot(quick_sort_data[:,:n], quick_sort_data[:,:μ], label="quicksort", 
        yscale=:log10, xscale=:log10, lw=3, c=:gray69, minorgrid=true, legend=:topleft)
    plot!(bubble_sort_data[:,:n], bubble_sort_data[:,:μ], label="bubblesort", 
        yscale=:log10, xscale=:log10, lw=3, c=:red)
    plot!(julia_sort_data[:,:n], julia_sort_data[:,:μ], label="Julia sort", 
        yscale=:log10, xscale=:log10, lw=3, c=:blue)
    xlims!(1e+0, 1.5e+3) # data runs to 2^10 = 1024
    ylims!(1e+0, 1e+7)
    xlabel!("Number of elements n", fontsize=18)
    ylabel!("Mean Runtime (ns)", fontsize=18)
end

___

## Summary
Completing bubble sort by hand, then timing three sorting methods on random data of the same sizes and distribution, separates how an algorithm scales from how fast it actually runs.

> __Key Takeaways:__
>
> * __Correctness comes before speed:__ An implementation is worth benchmarking only once it reproduces a reference result, since a timing number taken from wrong code measures nothing. Testing before timing is what makes a runtime comparison mean what it appears to mean.
> * __Growth rate and constant factor are separate claims:__ Two implementations can share a scaling exponent and still differ by a large constant multiple in runtime. Curves that look parallel on a log-log plot tell you the exponents match over the sizes you measured, not that the two are equally fast.
> * __Buying usually beats building, and not only on constants:__ A mature library routine wins partly through careful engineering and partly by selecting a different algorithm for the data it is handed, which can beat a hand-written implementation on growth rate rather than only on constants. A library author has already made choices you would otherwise have to discover yourself, which is the ordinary argument for buying.

Our bubblesort keeps pace with the library on the smallest inputs and then loses badly as the data grows, which is what an $\mathcal{O}(n^2)$ method looks like when you plot it and why production code should call the library even when you have written your own.
___